# ML Practical 3: Evaluation of multiple models [Solution for Part A]

## The task.

Task: Predict whether a person makes over $50k per year from census data known about them.

Data set from the paper: Kohavi, Ron. "Scaling up the accuracy of Naive-Bayes classifiers: a decision-tree hybrid." KDD. Vol. 96. 1996.
Data URL: We will be using modified versions of the publically avaliable data. Please download the data from the URLs provided.

In [1]:
import pandas as pd

# Output feature table
output_feature = pd.DataFrame({
    "Feature": ["salary"],
    "Type": ["categorical"],
    "Values": [">50K, <=50K"]
})

output_feature

,Feature,Type,Values
0,salary,categorical,">50K, <=50K"


In [2]:
# Input feature table
input_features = pd.DataFrame({
    "Feature": [
        "age",
        "workclass",
        "fnlwgt",
        "education",
        "education-num",
        "marital-status",
        "occupation",
        "relationship",
        "race",
        "sex",
        "capital-gain",
        "capital-loss",
        "hours-per-week",
        "native-country"
    ],
    "Type": [
        "continuous",
        "categorical",
        "continuous",
        "categorical",
        "continuous",
        "categorical",
        "categorical",
        "categorical",
        "categorical",
        "categorical",
        "continuous",
        "continuous",
        "continuous",
        "categorical"
    ],
    "Values": [
        "",
        "Private, Self-emp-not-inc, Self-emp-inc, Federal-gov, Local-gov, State-gov, Without-pay, Never-worked",
        "",
        "Bachelors, Some-college, 11th, HS-grad, Prof-school, Assoc-acdm, Assoc-voc, 9th, 7th-8th, 12th, Masters, 1st-4th, 10th, Doctorate, 5th-6th, Preschool",
        "",
        "Married-civ-spouse, Divorced, Never-married, Separated, Widowed, Married-spouse-absent, Married-AF-spouse",
        "Tech-support, Craft-repair, Other-service, Sales, Exec-managerial, Prof-specialty, Handlers-cleaners, Machine-op-inspct, Adm-clerical, Farming-fishing, Transport-moving, Priv-house-serv, Protective-serv, Armed-Forces",
        "Wife, Own-child, Husband, Not-in-family, Other-relative, Unmarried",
        "White, Asian-Pac-Islander, Amer-Indian-Eskimo, Other, Black",
        "Female, Male",
        "",
        "",
        "",
        "United-States, Cambodia, England, Puerto-Rico, Canada, Germany, Outlying-US(Guam-USVI-etc), India, Japan, Greece, South, China, Cuba, Iran, Honduras, Philippines, Italy, Poland, Jamaica, Vietnam, Mexico, Portugal, Ireland, France, Dominican-Republic, Laos, Ecuador, Taiwan, Haiti, Columbia, Hungary, Guatemala, Nicaragua, Scotland, Thailand, Yugoslavia, El-Salvador, Trinadad&Tobago, Peru, Hong, Holand-Netherlands"
    ]
})

input_features

,Feature,Type,Values
0,age,continuous,
1,workclass,categorical,"Private, Self-emp-not-inc, Self-emp-inc, Feder..."
2,fnlwgt,continuous,
3,education,categorical,"Bachelors, Some-college, 11th, HS-grad, Prof-s..."
4,education-num,continuous,
5,marital-status,categorical,"Married-civ-spouse, Divorced, Never-married, S..."
6,occupation,categorical,"Tech-support, Craft-repair, Other-service, Sal..."
7,relationship,categorical,"Wife, Own-child, Husband, Not-in-family, Other..."
8,race,categorical,"White, Asian-Pac-Islander, Amer-Indian-Eskimo,..."
9,sex,categorical,"Female, Male"


In [3]:
input_features[["Feature", "Type"]]

,Feature,Type
0,age,continuous
1,workclass,categorical
2,fnlwgt,continuous
3,education,categorical
4,education-num,continuous
5,marital-status,categorical
6,occupation,categorical
7,relationship,categorical
8,race,categorical
9,sex,categorical


In [4]:
all_features = pd.concat([
    input_features.assign(Role="Input"),
    output_feature.assign(Role="Output")
], ignore_index=True)

all_features = all_features[["Role", "Feature", "Type", "Values"]]
all_features

,Role,Feature,Type,Values
0,Input,age,continuous,
1,Input,workclass,categorical,"Private, Self-emp-not-inc, Self-emp-inc, Feder..."
2,Input,fnlwgt,continuous,
3,Input,education,categorical,"Bachelors, Some-college, 11th, HS-grad, Prof-s..."
4,Input,education-num,continuous,
5,Input,marital-status,categorical,"Married-civ-spouse, Divorced, Never-married, S..."
6,Input,occupation,categorical,"Tech-support, Craft-repair, Other-service, Sal..."
7,Input,relationship,categorical,"Wife, Own-child, Husband, Not-in-family, Other..."
8,Input,race,categorical,"White, Asian-Pac-Islander, Amer-Indian-Eskimo,..."
9,Input,sex,categorical,"Female, Male"


# To help you along some of the basic data preparation has been done for you.
Read the code. Understand what has been done.

In [ ]:
# Some basic imports
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
#from category_encoders.one_hot import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

# Read the data into a pandas DataFrame
data = pd.read_csv('https://drive.google.com/uc?export=download&id=1lBiNrYBk5KdfBllyjuELgRwYaK4yT2z9', header = 0, names = ['age','workclass','fnlwgt','education','education-num','matrial-status','occupation','relationship','race','sex','captial-gain','captial-loss','hours-per-week','salary'])

In [ ]:
# Define our input features and our output feature
# Call our input features X and our output feature y (the sklearn standard)
# Note that we have categorical features.
X = data.drop( columns = 'salary' )
y = data.salary

In [ ]:
# Now we need to encode our output feature to be an integer 0 or 1.
# This is because we have a binary classification problem and in order to use sklearn's
# built-in evaluation measures we need to have one class defined as 1 (target) and one as 0 (non-target).

# We could do this by using the LabelEncoder from sklearn. The LabelEncoder will convert n-distinct values
# to 0,..,n-1 values in this case giving us what we want. We assume that our training set contains both
# labels and that this mapping will be valid. However, we have no control
# over which value is represented by 1 and which is represented by 0.
# Therefore it is easier (in terms of subsequent interpretation) to do this
# manually. Recall the problem, we want our target variable (1) to be '>50k'

# To do this (your variable y is a pandas.Series object, use the replace method):
# 1) update all values '<=50K' within y to equal 0
# 2) update all values '>50K' within y to equal 1

y.replace(to_replace = ' <=50K', value = 0, inplace = True)
y.replace(to_replace = ' >50K', value = 1, inplace = True)

<ipython-input-3-62a5c3a64a6d>:17: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y.replace(to_replace = ' >50K', value = 1, inplace = True)


In [ ]:
# The baseline classifier for you to use
lr_model = Pipeline([
    ('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output = False)),  # will automatically pick string columns (could have specified)
    ('standardize', StandardScaler()), # will convert everything (can't specify which columns but all columns are fine after onehot)
    ('model',LogisticRegression(solver = 'liblinear') )
    ])



# Your turn. See the instructions in the slide deck...

In [ ]:
list_of_models = [lr_model]

In [ ]:
from sklearn.svm import LinearSVC

m1 = Pipeline( [
    ('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output = False)),  # will automatically pick string columns (could have specified)
    ('standardize', StandardScaler()), # will convert everything (can't specify which columns but all columns are fine)
    ('model',LinearSVC(max_iter=30000, C = 0.001) )
] )

m2 = Pipeline( [
    ('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output = False)),  # will automatically pick string columns (could have specified)
    ('standardize', StandardScaler()), # will convert everything (can't specify which columns but all columns are fine)
    ('model',LinearSVC(max_iter=30000, C = 0.01) )
] )

m3 = Pipeline( [
    ('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output = False)),  # will automatically pick string columns (could have specified)
    ('standardize', StandardScaler()), # will convert everything (can't specify which columns but all columns are fine)
    ('model',LinearSVC(max_iter=30000, C = 1) )
] )

m4 = Pipeline( [
    ('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output = False)),  # will automatically pick string columns (could have specified)
    ('standardize', StandardScaler()), # will convert everything (can't specify which columns but all columns are fine)
    ('model',LinearSVC(max_iter=30000, C = 100) )
] )

m5 = Pipeline( [
    ('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output = False)),  # will automatically pick string columns (could have specified)
    ('standardize', StandardScaler()), # will convert everything (can't specify which columns but all columns are fine)
    ('model',LinearSVC(max_iter=30000, C = 1000) )
] )

list_of_models += [m1,m2,m3,m4,m5]

In [ ]:
# Initial split
X_train, X_test, y_train, y_test =   train_test_split(X,y)

results = []

# For each model further split and do the evaluation
for model in list_of_models:
    # Define the further data split we'll use
    X_subtrain, X_valid, y_subtrain, y_valid = train_test_split( X_train, y_train, random_state = 42)

    # Train and test with this split

    model.fit(X_subtrain,y_subtrain)
    r = model.score(X_valid, y_valid)

    # Store the result
    results.append(r)

# Get the best model
idx_of_best_model = results.index(max(results))
best_model = list_of_models[idx_of_best_model]

# Train and test with the final split
best_model.fit(X_train, y_train)
score = best_model.score(X_test, y_test)

# Train deployment version
best_model.fit(X, y)


Pipeline(steps=[('onehot',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
                ('standardize', StandardScaler()),
                ('model', LogisticRegression(solver='liblinear'))])
Pipeline(steps=[('onehot',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
                ('standardize', StandardScaler()),
                ('model', LinearSVC(C=0.001, max_iter=30000))])
Pipeline(steps=[('onehot',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
                ('standardize', StandardScaler()),
                ('model', LinearSVC(C=0.01, max_iter=30000))])
Pipeline(steps=[('onehot',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
                ('standardize', StandardScaler()),
                ('model', LinearSVC(C=1, max_iter=30000))])
Pipeline(steps=[('onehot',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
           

Pipeline(steps=[('onehot',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
                ('standardize', StandardScaler()),
                ('model', LinearSVC(C=0.001, max_iter=30000))])

In [ ]:
best_model

Pipeline(steps=[('onehot',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
                ('standardize', StandardScaler()),
                ('model', LinearSVC(C=0.001, max_iter=30000))])

In [ ]:
score

0.84

In [ ]:
results

[0.8356164383561644,
 0.8508371385083714,
 0.8249619482496194,
 0.7884322678843226,
 0.7808219178082192,
 0.7808219178082192]